# 07 — Advanced identification and solver routing (optional)

**Audience:** users debugging hard graph problems or reading `tight=False`
warnings. Several APIs here are **experimental** (`classify_program`,
`is_identified`, `identify_query`) and may evolve on the 0.x train.

Prefer `method="auto"` in applications. Use this notebook to understand *why*
a result looks the way it does.


In [1]:
import numpy as np
from partialid import (
    Assumptions, CausalDiagram, Query, Variables, bound, classify_program,
    identify_query, is_identified,
)

def softmax_P(V, seed=0):
    rng = np.random.default_rng(seed)
    keys = V.cells()
    logits = rng.normal(size=len(keys))
    w = np.exp(logits - logits.max())
    w = w / w.sum()
    return {k: float(v) for k, v in zip(keys, w)}


## Point ID vs partial ID

Back-door (observed confounder) and front-door can be point-identified.
Order-only Manski on the same variables is not.


In [2]:
V = Variables.from_binary("Z", "X", "Y")
g_bd = CausalDiagram.from_edges(
    ["Z", "X", "Y"],
    directed=[("Z", "X"), ("Z", "Y"), ("X", "Y")],
    bidirected=[],
)
P = softmax_P(V, seed=0)
Q = Query.interventional(outcome="Y", do={"X": 1}, value=1)
a_bd = Assumptions.from_diagram(g_bd)
print("back-door identified?", is_identified(g_bd, ["X"], ["Y"]))
print(identify_query(V, P, Q, a_bd))
r_bd = bound(V, P, Q, a_bd)
print(r_bd.summary(Q))

r_order = bound(V, P, Q, Assumptions.order("Z", "X", "Y"))
print(f"order-only width {r_order.width:.4f} >> ID width {r_bd.width:.4f}")


back-door identified? True
IDResult(identified=True, value=0.3956808162722366, criterion='id', relevant_components=[], notes=["C*=[['Z'], ['X'], ['Y']]", 'point-identified by Shpitser–Pearl ID'], attempted=True)
query: P(Y=1 | do(X=1))
interval: [0.395681, 0.395681]
width: 0
method: id
tight: True
success: True
message: point-identified
order-only width 0.3031 >> ID width 0.0000


In [3]:
V_fd = Variables.from_binary("X", "M", "Y")
g_fd = CausalDiagram.from_edges(
    ["X", "M", "Y"],
    directed=[("X", "M"), ("M", "Y")],
    bidirected=[("X", "Y")],
)
P_fd = softmax_P(V_fd, seed=1)
Q_fd = Query.interventional(outcome="Y", do={"X": 1}, value=1)
r_fd = bound(V_fd, P_fd, Q_fd, Assumptions.from_diagram(g_fd))
print(r_fd.summary(Q_fd))
print("classify:", classify_program(g_fd, query=Q_fd, variables=V_fd))


query: P(Y=1 | do(X=1))
interval: [0.50148, 0.50148]
width: 0
method: id
tight: True
success: True
message: point-identified
classify: ProgramDegree(kind=<ProgramKind.POINT: 'point'>, degree=0, free_districts=(), c_degrees=(), notes=("ancestral=['M', 'X', 'Y']", "districts=[['X', 'Y'], ['M']]", "c_degrees={('X', 'Y'): 1, ('M',): 1}", "free_districts=[['X', 'Y']]", "single free district ['X', 'Y']; exo=['M']", 'program: point (ID)'), wl_wr_cut=None, in_hypothesis_H=True, pre_cut_free_districts=(frozenset({'X', 'Y'}),))


## `classify_program` (experimental)

Returns a `ProgramDegree` with `kind` ∈ {point, linear, bilinear/…, polynomial}
and a numeric `degree`. This is the signal `bound(..., method="auto")` uses for
routing.


In [4]:
for label, g in [
    ("back-door", g_bd),
    ("front-door", g_fd),
    ("bow (X↔Y, X→Y)", CausalDiagram.from_edges(
        ["X", "Y"], directed=[("X", "Y")], bidirected=[("X", "Y")],
    )),
]:
    Vloc = Variables.from_binary(*g.nodes)
    Qi = Query.interventional(outcome="Y", do={"X": 1}, value=1)
    prog = classify_program(g, query=Qi, variables=Vloc)
    print(f"{label:20s}  kind={prog.kind.value:18s}  degree={prog.degree}")


back-door             kind=point               degree=0
front-door            kind=point               degree=0
bow (X↔Y, X→Y)        kind=linear              degree=1


## Outer default for degree ≥ 3

For polynomial programs, `bound` defaults to a merged-latent **outer** LP
(`tight=False`). Opt into nested multilinear search with `prefer_poly=True`
(more expensive; still check `tight`).


In [5]:
V_t = Variables.from_binary("X", "Y", "Z")
g_t = CausalDiagram.from_edges(
    ["X", "Y", "Z"],
    directed=[("X", "Y"), ("Y", "Z"), ("X", "Z")],
    bidirected=[("X", "Y"), ("Y", "Z"), ("X", "Z")],
)
P_t = {k: 1.0 / 8 for k in V_t.cells()}
Q_t = Query.interventional(outcome="Y", do={"X": 1}, value=1)
prog_t = classify_program(g_t, query=Q_t, variables=V_t)
print("classify:", prog_t.kind, "degree=", prog_t.degree)

r_outer = bound(V_t, P_t, Q_t, Assumptions.from_diagram(g_t))
print("default:", r_outer.summary(Q_t))
print("tight=", r_outer.tight, "warnings=", r_outer.warnings)

r_poly = bound(
    V_t, P_t, Q_t, Assumptions.from_diagram(g_t), prefer_poly=True,
)
print("prefer_poly:", r_poly.summary(Q_t))
print("tight=", r_poly.tight, "method=", r_poly.method)


classify: ProgramKind.LINEAR degree= 1
default: query: P(Y=1 | do(X=1))
interval: [0.25, 0.75]
width: 0.5
method: fiber
tight: True
success: True
tight= True warnings= []
prefer_poly: query: P(Y=1 | do(X=1))
interval: [0.25, 0.75]
width: 0.5
method: fiber
tight: True
success: True
tight= True method= fiber


## Exclusions and latent hyperedges

Missing directed edges are first-class via `Assumptions.exclusion`. Latent
confounding is encoded as bidirected / hyperedges on `CausalDiagram`.


In [6]:
# Order X→M→Y with confounding X↔Y, plus an explicit exclusion already in the diagram.
# Compare to adding an exclusion atom on a denser baseline graph.
V = Variables.from_binary("X", "M", "Y")
P = softmax_P(V, seed=2)
Q = Query.interventional(outcome="Y", do={"X": 1}, value=1)

g_dense = CausalDiagram.from_edges(
    ["X", "M", "Y"],
    directed=[("X", "M"), ("M", "Y"), ("X", "Y")],
    bidirected=[("X", "Y")],
)
# Drop the direct X→Y edge via exclusion
a_ex = Assumptions.from_diagram(g_dense) & Assumptions.exclusion("Y", "X")
r_ex = bound(V, P, Q, a_ex)
print(r_ex.summary(Q))
print("tight=", r_ex.tight)


query: P(Y=1 | do(X=1))
interval: [0.362864, 0.537169]
width: 0.174304
method: fiber
tight: True
success: True
tight= True


## Trust contract

1. Prefer **`method="auto"`**; treat explicit solver names as escape hatches.
2. Always read **`tight`** and **`warnings`**.
3. Deg ≥ 3 outer approximations are valid but conservative unless you opt into
   `prefer_poly=True`.
4. Experimental classifiers and ID helpers help explain routing — they are not
   required for ordinary `bound` workflows.

You have finished the optional advanced track. Return to the core curriculum
([01](01_first_bounds.ipynb)–[05](05_queries_and_results.ipynb)) for applied
analyses, or continue with the 8-variable panel in
[06_panel_and_scale.ipynb](06_panel_and_scale.ipynb).
